# SPIN-Soft — Step 1/4: importance -> coupled indices (ONE notebook, all models)

**Identification is identical to regular SPIN — α does NOT appear in this step.** SPIN-Soft's only
change to SPIN (Algorithm 1, line 13) is *how* coupled weights are suppressed (`αW` instead of `0`),
and that happens in Steps 2–4. The *set* of coupled weights is found exactly as in the faithful 7B
Step 1: importance `|W|·Σ|∇L|` (Michel et al. 2019) + the authors' **exact** `SPIN.py::get_set_difference_mask`
positional index-mask (the released code, not the paper prose — replicated for result-fidelity).

Run this notebook **once per model**: uncomment one line in the model-select block, run all. It is
checkpointed per layer-group (`coupled_ckpt_<tag>.pt`), so a crash/timeout resumes.

**Output:** `coupled_soft_<tag>.json` — feeds Step 2 (HSIC+ppl) and Step 3 (generate).
**Runtime:** GPU **T4 x2**, Internet ON. ~5–20 min (POC) / ~20 min (7B).

> Sanity: identification == regular SPIN, so for `Qwen2-7B-Instruct` this file should match the
> existing `7B testing/Results/coupled_indices.json` (and `coupled_iter_*_n1.json`).

## Step 0 — env + config + **model select**

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import random, gc, time, json
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# SELECT MODEL — uncomment EXACTLY ONE line.
# Each line sets MODEL_ID, p, q, USE_FAST together so the per-model
# extraction ratio and tokenizer can never drift apart.
# (values lifted verbatim from your Iter-SPIN Step-1 configs)
# ------------------------------------------------------------
#MODEL_ID,P,Q,USE_FAST = 'google/gemma-2-2b-it',            5e-7, 5e-7, False   # POC (gated: HF token+license)
# MODEL_ID,P,Q,USE_FAST = 'meta-llama/Llama-3.2-3B-Instruct', 1e-6, 1e-6, True   # POC (gated; NOTE p=q=1e-6 AND fast tokenizer)
# MODEL_ID,P,Q,USE_FAST = 'Qwen/Qwen2.5-3B-Instruct',        5e-7, 5e-7, False   # POC
MODEL_ID,P,Q,USE_FAST = 'Qwen/Qwen2-7B-Instruct',          5e-7, 5e-7, False   # scale-up
# ============================================================
NSAMPLES=128
TARGET='mlp'
MAXLEN=1024          # generous cap (authors don't truncate; their samples are short). Safety only.
LAYER_GROUP=4        # score 4 layers at a time (memory); scores are identical to per-layer.
SEED=0
DTYPE=torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_TAG=MODEL_ID.split('/')[-1]
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
print('SoftSPIN Step 1 | model', MODEL_ID, '(tag', MODEL_TAG+') | group', LAYER_GROUP, '| p=q', P)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.9 MB/s eta 0:00:00
SoftSPIN Step 1 | model Qwen/Qwen2-7B-Instruct (tag Qwen2-7B-Instruct) | group 4 | p=q 5e-07


## Step 0b — Hugging Face auth (gemma-2 & Llama-3.2 are gated)

On Kaggle: add your HF token under **Add-ons → Secrets** as `HF_TOKEN`. Qwen models are open and
need no token.

In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    _tok=UserSecretsClient().get_secret('HF_TOKEN')
    from huggingface_hub import login; login(token=_tok); print('HF login OK (Kaggle secret HF_TOKEN)')
except Exception as e:
    print('No Kaggle HF_TOKEN secret picked up:', repr(e))
    print('  -> fine for Qwen (open). For gemma/llama, set the secret or run huggingface_hub.login() manually.')

No Kaggle HF_TOKEN secret picked up: BackendError("Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 133043513 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.")
  -> fine for Qwen (open). For gemma/llama, set the secret or run huggingface_hub.login() manually.


## Step 1 — data + model

In [3]:
DEAN_DIR='/tmp/DEAN'   # clone to scratch, NOT /kaggle/working, so DEAN is never saved to notebook output
if not os.path.exists(DEAN_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git',DEAN_DIR], check=True)
DATA=f'{DEAN_DIR}/src/data'
def load_pr(p): d=pd.read_csv(p); return d['prompt'].tolist(), d['response'].tolist()
RAW={'g':load_pr(f'{DATA}/alpaca_cleaned_no_safety_train.csv'),      # general capability
     'f':load_pr(f'{DATA}/beaver_train330k_fairness_safe_1k.csv'),   # fairness
     'p':load_pr(f'{DATA}/beaver_train330k_privacy_safe_1k.csv')}    # privacy

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})  # cap sized for 7B on T4x2; harmless for POC
model.eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]
def make_batches(pr_rs,n):
    # utils.get_loaders(disentangle=True): prompt + response[:,1:] (UNCONDITIONAL drop, authors' code),
    # target = input with prompt tokens masked to -100.
    prompts,resp=pr_rs; idx=list(range(len(prompts))); random.Random(SEED).shuffle(idx); idx=idx[:n]; out=[]
    for i in idx:
        ep=tokenizer(str(prompts[i]),return_tensors='pt').input_ids
        er=tokenizer(str(resp[i]),return_tensors='pt').input_ids[:,1:]   # authors' unconditional drop
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={k:make_batches(RAW[k],NSAMPLES) for k in RAW}
print('calibration batches:', {k:len(v) for k,v in BATCHES.items()})

Cloning into '/tmp/DEAN'...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

model loaded | entry cuda:0
  [loaded] GPU0 8.8/15.6 GPU1 7.0/15.6 GiB free
calibration batches: {'g': 127, 'f': 128, 'p': 128}


## Step 2 — score group-by-group, derive coupled indices, save only those

Authors' exact selection: coupled = (top-p by importance on general ∩ complements) — the
`get_set_difference_mask` positional logic, verbatim. Only the coupled indices are saved (full
importance tensors are ~GBs/dataset and overflow disk).

In [4]:
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
CK=f'/kaggle/working/coupled_ckpt_{MODEL_TAG}.pt'   # tag-specific: switching models never wrong-resumes

if os.path.exists(CK):
    st=torch.load(CK); coupled=st['coupled']; start=st['next_group']; print('resuming from group', start)
else:
    coupled={}; start=0

# ---- authors' EXACT SPIN.py logic (verbatim) ----
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p = int(p * W_metric1.shape[1] * W_metric1.shape[0])
    top_q = int(q * W_metric2.shape[1] * W_metric2.shape[0])
    top_p_indices = torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices = torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p = torch.unique(top_p_indices)
    unique_q = torch.unique(top_q_indices)
    mask_only_metric1 = ~torch.isin(unique_p, unique_q)
    mask_only_metric2 = ~torch.isin(unique_q, unique_p)
    mask_intersection = ~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q

def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, mask_only_trust1, _, unique_trust1_indices = get_set_difference_mask(P, Q, W_general, W_fair)
    _, mask_only_trust2, _, unique_trust2_indices = get_set_difference_mask(P, Q, W_general, W_priv)
    mask_trust_intersection = mask_only_trust1 & mask_only_trust2
    filtered_indices = unique_trust1_indices[mask_trust_intersection]
    return filtered_indices

@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}  # to CPU, free GPU
    del acc; gc.collect(); torch.cuda.empty_cache()
    return imp

model.gradient_checkpointing_enable(); model.config.use_cache=False
for gi in range(start, len(GROUPS)):
    grp=GROUPS[gi]; gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
    t0=time.time()
    imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
    for n in gnames:
        fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])   # authors' exact selection
        if len(fi): coupled[n]=[int(x) for x in fi.tolist()]
    del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
    torch.save({'coupled':coupled,'next_group':gi+1}, CK)
    tot=sum(len(v) for v in coupled.values())
    print(f'  group {gi+1}/{len(GROUPS)} done ({time.time()-t0:.0f}s) | coupled so far {tot}'); memline()
model.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad_(False)

meta={'model':MODEL_ID,'tag':MODEL_TAG,'variant':'soft-SPIN (identification = regular SPIN one-shot)',
      'nsamples':NSAMPLES,'target':TARGET,'p':P,'q':Q,
      'coupled_count':sum(len(v) for v in coupled.values()), 'coupled':coupled}
OUT=f'/kaggle/working/coupled_soft_{MODEL_TAG}.json'
json.dump(meta, open(OUT,'w'))
print(f'\nSTEP 1 DONE ({MODEL_TAG}) -> {OUT} | coupled weights:', meta['coupled_count'])
print('This coupled set feeds Steps 2 (HSIC+ppl) and 3 (generate). ALPHA enters THERE, not here.')

  group 1/7 done (161s) | coupled so far 248
  [] GPU0 7.1/15.6 GPU1 6.9/15.6 GiB free
  group 2/7 done (157s) | coupled so far 510
  [] GPU0 7.1/15.6 GPU1 6.9/15.6 GiB free
  group 3/7 done (156s) | coupled so far 647
  [] GPU0 7.1/15.6 GPU1 6.9/15.6 GiB free
  group 4/7 done (156s) | coupled so far 767
  [] GPU0 8.8/15.6 GPU1 5.2/15.6 GiB free
  group 5/7 done (148s) | coupled so far 883
  [] GPU0 8.8/15.6 GPU1 5.2/15.6 GiB free
  group 6/7 done (144s) | coupled so far 1010
  [] GPU0 8.8/15.6 GPU1 5.2/15.6 GiB free
  group 7/7 done (139s) | coupled so far 1148
  [] GPU0 8.8/15.6 GPU1 5.2/15.6 GiB free

STEP 1 DONE (Qwen2-7B-Instruct) -> /kaggle/working/coupled_soft_Qwen2-7B-Instruct.json | coupled weights: 1148
This coupled set feeds Steps 2 (HSIC+ppl) and 3 (generate). ALPHA enters THERE, not here.
